In [7]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

#####################################
# Chapter 2: 데이터 로딩 및 처리
#####################################
"""
입력 텍스트 ("Hello, I am")
    ↓
[토크나이저] → 토큰 ID 배열
    ↓
┌─────────────────────────────────────┐
│           GPTModel                  │
│                                     │
│  tok_emb  → 토큰 임베딩             │
│  pos_emb  → 위치 임베딩             │
│     +  (합산)                       │
│  drop_emb → 드롭아웃                │
│     ↓                               │
│  ┌─────────────────────┐  × 12      │
│  │  TransformerBlock   │            │
│  │  LayerNorm          │            │ 층 정규화: 학습 중 각 층의 출력값이 너무 크거나 작아지는 현상 방지, 학습 안정성 향상
│  │  MultiHeadAttention │            │
│  │  + Residual         │            │
│  │  LayerNorm          │            │ 층 정규화
│  │  FeedForward        │            │
│  │  + Residual         │            │
│  └─────────────────────┘            │
│  final_norm → LayerNorm             │
│  out_head   → Linear(768→50257)     │
└─────────────────────────────────────┘
    ↓
logits → argmax → 다음 토큰 예측
"""

class GPTDatasetV1(Dataset):
    """
    GPT 학습을 위한 데이터셋 클래스입니다.
    텍스트를 입력받아 토큰화하고, 입력(input)과 타겟(target) 쌍을 만듭니다.
    """
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 1. 전체 텍스트를 토큰화합니다.
        # <|endoftext|> 같은 특수 토큰도 허용하여 인코딩합니다.
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # 2. 슬라이딩 윈도우 방식으로 데이터를 조각냅니다.
        # stride만큼 이동하면서 max_length 길이의 덩어리(chunk)를 만듭니다.
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            # 타겟은 입력보다 한 칸 뒤의 토큰들입니다 (다음 토큰 예측 과제).
            target_chunk = token_ids[i + 1: i + max_length + 1]
            
            # 텐서로 변환하여 저장
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # 데이터셋의 총 샘플 수 반환
        return len(self.input_ids)

    def __getitem__(self, idx):
        # 특정 인덱스의 입력과 타겟 쌍 반환
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    """
    텍스트 데이터를 받아 학습에 사용할 DataLoader를 생성하는 함수입니다.
    """
    # 토크나이저 초기화 (GPT-2용 BPE 인코딩 사용)
    tokenizer = tiktoken.get_encoding("gpt2")

    # 데이터셋 생성
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 데이터로더 생성 (배치 단위로 데이터를 묶어주고 셔플링 수행)
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


In [8]:

#####################################
# Chapter 3: 어텐션 메커니즘 (모델의 핵심)
#####################################
class MultiHeadAttention(nn.Module):
    """
    멀티 헤드 셀프 어텐션 (Multi-Head Self-Attention) 모듈입니다.
    입력 데이터 간의 관계성을 여러 관점(Head)에서 병렬로 학습합니다.
    시퀀스 내 토큰들 사이의 관계와 문맥을 여러 관점(Head)에서 동시에 학습
    """
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "출력 차원(d_out)은 헤드 수(num_heads)로 나누어 떨어져야 합니다."

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # 각 헤드가 담당할 차원 크기, d_out=768, num_heads=12이면, head_dim=768/12 = 64

        # Query, Key, Value 변환 행렬 (각각 독립적인 학습 파라미터)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        # 모든 헤드 결과를 하나로 합친 후 통과시키는 출력 레이어
        self.out_proj = nn.Linear(d_out, d_out) 
        self.dropout = nn.Dropout(dropout)
        
        # Causal Mask (인과적 마스킹) 생성: 미래의 토큰을 보지 못하게 함
        # [[0, 1, 1],   ← 1인 위치는 -inf로 마스킹됨
        #  [0, 0, 1],
        #  [0, 0, 0]]
        # 상삼각 행렬(대각선 위쪽)을 1로 채워서 나중에 마스킹에 사용
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape # b: 배치 크기, num_tokens: 시퀀스 길이

        # ── Step 1: Q, K, V 계산 ─────────────────────────────
        keys = self.W_key(x)      # Shape: (b, num_tokens, d_out).. 즉 (1, 5, 768)
        queries = self.W_query(x) # {1, 5, 768)
        values = self.W_value(x)  # {1, 5, 768)

        # ── Step 2: 헤드별로 분할 ─────────────────────────────
        # 차원을 변형하여 여러 헤드가 병렬로 처리할 수 있게 함
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # ── Step 3: 차원 순서 변경 ────────────────────────────
        # (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        # 이렇게 하면 (num_tokens, head_dim) 행렬이 헤드 개수만큼 독립적으로 존재하게 됨
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # ── Step 4: 어텐션 점수 계산 (유사도) ────────────────
        # Scaled Dot-Product Attention 계산 (Query와 Key의 내적)
        attn_scores = queries @ keys.transpose(2, 3)  # 결과 Shape: (b, num_heads, num_tokens, num_tokens)

        # ── Step 5: 인과적 마스킹 (미래 토큰 차단) ───────────
        # 현재 시점보다 미래의 토큰 정보를 참조하지 못하게 -inf로 가림
        # 미래 토큰을 보면, "답을 미리 보는" 것이 되므로 차단함!!!
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # ── Step 6: Softmax + Dropout ─────────────────────────        
        # 점수를 확률로 변환 (합이 1이 되도록), √head_dim으로 스케일링 → 점수가 너무 커지는 것 방지
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # ── Step 7: Value와의 가중합 (Context Vector) ─────────
        # Shape: (b, num_heads, num_tokens, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 

        # ── Step 8: 헤드 결합 ─────────────────────────────────
        # 나눠졌던 헤드들을 다시 원래의 d_out 차원으로 합침
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        
        # ── Step 9: 최종 선형 투영 ────────────────────────────
        context_vec = self.out_proj(context_vec) 

        return context_vec


In [9]:

#####################################
# Chapter 4: GPT 아키텍처 구성 요소
#####################################
class LayerNorm(nn.Module):
    """
    층 정규화 (Layer Normalization): 학습 안정성을 높임 [SG] p.103 !!!!!!!
    """
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5   # [SG] 분모가 0이 되는것 방지하는 아주 작은 값
        self.scale = nn.Parameter(torch.ones(emb_dim)) # 학습 가능한 스케일 파라미터 (Gamma): 정규화 후 스케일 조정
        self.shift = nn.Parameter(torch.zeros(emb_dim)) # 학습 가능한 시프트 파라미터 (Beta): 정규화 후 이동

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True) # [SG] 마지막 차원(emb_dim)으로 평균
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        
        # TODO: LayerNorm 계산식의 통계량 변수를 채우세요.
        # 힌트: 평균(mean)과 분산(var)을 사용해 정규화합니다.
        # norm_x = (x - ????) / torch.????(??? + self.eps)
        norm_x = (x - mean) / torch.sqrt(var + self.eps) # [SG] 정규화: 평균을 빼고 표준편차로 나눔
        
        # TODO: LayerNorm 학습 파라미터를 채우세요.
        # 힌트: scale로 곱하고 shift를 더하는 형태를 사용합니다.
        # return self.???? * ???? + self.????
        return self.scale * norm_x + self.shift # [SG] 학습 가능 파라미터로 복원력 부여


class GELU(nn.Module):
    """
    GELU 활성화 함수: GPT 계열에서 주로 사용하는 비선형 함수
    """
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    """
    피드 포워드 네트워크 (Feed-Forward Network)
    어텐션이 모은 정보를 각 토큰별로 개별적으로 가공하는 역할
    보통 임베딩 차원을 4배로 늘렸다가 다시 줄임
    """
    def __init__(self, cfg):
        super().__init__()
        # TODO: FeedForward의 확장 배수와 활성화 함수 클래스를 채우세요.
        # 힌트: GPT 구조에서는 hidden 크기를 4배로 확장합니다. 확장 Linear와 축소 Linear 사이에 GELU 활성화 함수를 넣으세요.
        self.layers = nn.Sequential(            
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), # [SG] 1단계: embedding 차원을 4배로 확장 (768 -> 3072)
            # ????,
            GELU(), # [SG] 2단계: 비선형 활성화 함수
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), # [SG] 3단계: 다시 원래대로 embedding 차원을 압축 (3072 -> 768)
        )

    def forward(self, x):
        return self.layers(x)


### 차원 변화 시각화
"""
(batch, seq_len, 768)
        ↓  Linear
(batch, seq_len, 3072)   ← 4배로 팽창 (더 넓은 공간에서 특징 추출)
        ↓  GELU
(batch, seq_len, 3072)
        ↓  Linear
(batch, seq_len, 768)    ← 다시 원래 크기로 압축
"""

In [13]:
##중요 : 아래는 GPT 모델 전체 구조를 정의하는 코드입니다

#####################################
# Chapter 4: GPT 아키텍처 조립
#####################################
class TransformerBlock(nn.Module):
    """
    표준 트랜스포머 블록 (Decoder Block)
    구조: LayerNorm -> Attention -> Add(Residual) -> LayerNorm -> FeedForward -> Add(Residual)
    즉, Attention + FeedForward를 묶은 하나의 처리 단위임. GPT-2는 이것을 12개 쌓음.
    """
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"]) # Attension 전
        self.norm2 = LayerNorm(cfg["emb_dim"]) # FF 
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # ── 어텐션 블록 ────────────────────────────────────────
        # shortcut = x              # 원본 입력 저장 (잔차 연결용)
        # x = self.norm1(x)         # Pre-LayerNorm (정규화 먼저!)
        # x = self.att(x)           # Multi-Head Attention
        # x = self.drop_shortcut(x) # Dropout
        # x = x + shortcut          # ✅ 잔차 연결 (Residual Connection)

        # ── 피드 포워드 블록 ───────────────────────────────────
        # shortcut = x              # 업데이트된 x 저장
        # x = self.norm2(x)         # Pre-LayerNorm
        # x = self.ff(x)            # FeedForward
        # x = self.drop_shortcut(x) # Dropout
        # x = x + shortcut          # ✅ 잔차 연결
        
        # ====> 이런 구조임
        
        # region [어텐션 블록 (Residual Connection 적용)]
        shortcut = x ## [SG] 원래인풋을 저장 (잔차연결 위해서)
        x = self.norm1(x) # Pre-LayerNorm 방식
        # TODO
        # x = self.????(x)
        x = self.att(x) # [SG] Multi-Head Attention 수행
        
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력을 더해줌 (기울기 소실 방지)
        # endregion

        # region [피드 포워드 블록 (Residual Connection 적용)]
        shortcut = x
        x = self.norm2(x)
        # TODO
        # x = self.????(x)
        x = self.ff(x) # [SG] FeedForward 내부에서 이미 활성화 함수 적용
        
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력 다시 더함 [SG] 잔차연결!!!!
        # endregion

        return x
class GPTModel(nn.Module):
    """
    전체 GPT 모델 구조 정의
    Embedding -> Transformer Blocks -> Final Norm -> Output Head
    """
    def __init__(self, cfg):
        super().__init__()
        # 토큰 임베딩: 토큰 ID → 768차원 벡터
        # vocab_size(50257)개의 단어 각각에 768차원 벡터 할당        
        # TODO
        # self.tok_emb = nn.Embedding(cfg["????"], cfg["emb_dim"])
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"]) # [SG] tok_emb는 단어(token) embedding이므로 context_length가 아니고 vocab_size !!!!!!
        
        # 위치 임베딩: 위치 번호 → 768차원 벡터
        # context_length(1024)개의 위치 각각에 768차원 벡터 할당
        # TODO 
        # self.pos_emb = nn.Embedding(cfg["????"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"]) # [SG] pos_emb는 위치 embedding이므로 context_length 임!!!!!!
        
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # 트랜스포머 블록 쌓기 (n_layers 만큼)
        # 12개의 TransformerBlock을 순차적으로 연결
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]) ## [SG] !!!!!

        # 최종 정규화
        self.final_norm = LayerNorm(cfg["emb_dim"])
        
        # 최종 출력: 768차원 → 50257차원 (각 단어의 점수)
        # TODO
        # self.out_head = nn.Linear(cfg["emb_dim"], cfg["????"], bias=False)
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False) # [SG] embedding 차원 -> vocab 사이즈로 바뀜 (context_length가 아님!!!)77

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        # in_idx: (1, 5) — 토큰 ID들
        
        # 토큰 임베딩
        # (1, 5) → (1, 5, 768)
        tok_embeds = self.tok_emb(in_idx)
        
        # 위치 임베딩: [0, 1, 2, 3, 4] 위치 정보
        # (5,) → (5, 768)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))        

        # region [토큰 임베딩과 위치 임베딩 합산]
        # (1, 5, 768) + (5, 768) → (1, 5, 7
        # TODO
        # x = tok_embeds + ????
        x = tok_embeds + pos_embeds
        # endregion

        x = self.drop_emb(x)
        
        # region [트랜스포머 블록 통과]
        # TODO
        # x = self.????(x)
        x = self.trf_blocks(x) # [SG] 12개의 블록을 순차적으로 통과
        # endregion

        # region [최종 출력 계산]
        # (1, 5, 768) → (1, 5, 50257)
        x = self.final_norm(x)
        # TODO
        # logits = self.????(x) # 각 단어에 대한 예측 점수 (Logits)
        logits = self.out_head(x) # vocab_size 차원으로 변환함
        
        # endregion
        
        return logits

In [14]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    """
    간단한 텍스트 생성 루프
    현재 문맥을 넣어 다음 토큰을 예측하고, 이를 다시 문맥에 추가하여 반복함
    """
    # idx: 현재 문맥의 토큰 인덱스들 (Batch, Time)
    for _ in range(max_new_tokens):

        # 모델이 지원하는 최대 길이(context_size)를 넘지 않도록 자름
        # 즉, 1024 토큰 초과 시 뒤에서부터 자름
        idx_cond = idx[:, -context_size:]

        # 츠론 -> 모델 예측 (기울기 계산 불필요: 메모리 절약)
        # TODO
        # with torch.????():
        with torch.no_grad(): # [SG] 학습이 아니라 추론이니까 기울기 계산을 하면 안됨
            logits = model(idx_cond)

        # 마지막 위치의 예측값만 사용
        # (batch, n_token, vocab_size) -> (batch, vocab_size)
        # TODO
        # logits = logits[:, ????, :]
        logits = logits[:, -1, :] # [SG] 맨 마지막 sequence의 logit을 가져옴.

        # Greedy Decoding: 가장 점수 높은 토큰 선택
        # (1, 1)
        # TODO
        # idx_next = torch.????(logits, dim=-1, keepdim=True)
        idx_next = torch.argmax(logits, dim=-1, keepdim=True) # [SG] Greedy Decoding에서는 가장 큰 확률(logit)을 가진 token 선택!!!

        # Greedy Decoding의 특성
        # 현재 구현: Greedy (항상 가장 높은 확률 선택)
        # → 결정적(deterministic), 빠름, 하지만 단조로울 수 있음
        
        # 실제 GPT에서 사용하는 방식들:
        # Top-K Sampling   → 상위 K개 중에서 랜덤 선택
        # Top-p (Nucleus)  → 누적 확률 p% 이내에서 랜덤 선택
        # Temperature      → 확률 분포를 날카롭게/부드럽게 조절
        
        # 새 토큰을 시퀀스 뒤에 추가
        # (1, seq_len + 1)
        # TODO
        # idx = torch.????((idx, idx_next), dim=1) 
        idx = torch.cat((idx, idx_next), dim=1) # [SG] 새로 예측된 토큰을 현재 sequence 뒤에 이어 붙임

    return idx
    # (1, 원본길이 + max_new_tokens)


In [15]:
def main():
    # GPT-2 Small 모델 설정값 (124M 파라미터)
    GPT_CONFIG_124M = {
        "vocab_size": 50257,     # 단어 집합 크기
        "context_length": 1024,  # 최대 문맥 길이
        "emb_dim": 768,          # 임베딩 차원
        "n_heads": 12,           # 어텐션 헤드 수
        "n_layers": 12,          # 레이어 수
        "drop_rate": 0.1,        # 드롭아웃 비율
        "qkv_bias": False        # Q,K,V 편향 사용 여부
    }

    torch.manual_seed(123)
    
    # 모델 초기화 및 평가 모드 설정 (드롭아웃 비활성화)
    model = GPTModel(GPT_CONFIG_124M)
    model.eval() # [SG] dropout을 비활성화 시킴 !!!! 최초의 모델 추론시에는 normalization에서 beta, gamma가 변화면 안됨 (p.103)

    start_context = "Hello, I am"

    # 입력 텍스트 인코딩
    tokenizer = tiktoken.get_encoding("gpt2")
    encoded = tokenizer.encode(start_context)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # 배치 차원 추가 [SG] batch가 1인 data를 만듦 

    print(f"\n{50*'='}\n{22*' '}IN\n{50*'='}")
    print("\nInput text:", start_context)
    print("Encoded input text:", encoded)
    print("encoded_tensor.shape:", encoded_tensor.shape)

    # 텍스트 생성 실행
    out = generate_text_simple(
        model=model,
        idx=encoded_tensor,
        max_new_tokens=10,  # 10개의 새로운 토큰 생성
        context_size=GPT_CONFIG_124M["context_length"]
    )
    
    # 생성된 결과 디코딩
    decoded_text = tokenizer.decode(out.squeeze(0).tolist()) # [SG] 위에서 batch가 1인 data를 batch 1번 씌웠으므로 다시 하나 벗겨야 함

    print(f"\n\n{50*'='}\n{22*' '}OUT\n{50*'='}")
    print("\nOutput:", out)
    print("Output length:", len(out[0]))
    print("Output text:", decoded_text)

if __name__ == "__main__":
    main()


                      IN

Input text: Hello, I am
Encoded input text: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


                      OUT

Output: tensor([[15496,    11,   314,   716, 28391,  7283, 30769, 29279, 39520, 12912,
         46463, 39147,  7385, 50022]])
Output length: 14
Output text: Hello, I amyx IT Became1983Animation frozen Wilkinson shorth Kiralions
